# RG3 RH 미성형 8개: 변화 순서 및 누적 패턴
기존 순간값/평균편차와 구별되는 시간 경로 특징의 탐색. 미래 회복은 설명 전용, 모델 학습 없이 정상과 비교한다.

In [1]:
from pathlib import Path
import json, subprocess
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
assert subprocess.check_output(['git','branch','--show-current'],text=True).strip()=='hayeon'
old=json.loads((root/'notebooks/10_original_defect_report_revision.ipynb').read_text(encoding='utf-8'))
exec(''.join(next(c for c in old['cells'] if c['cell_type']=='code')['source']).split('for feature in features:')[0])
shot=pd.read_csv(tables/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=shot.loc[shot['product'].eq('RG3')&(shot.PassOrFail.eq(0)|shot.Reason.str.contains('미성형',na=False))].sort_values('TimeStamp').reset_index(drop=True)
rh=labels.loc[labels['product'].eq('RG3')&labels.side.eq('RH')]
check=d[keys+['PassOrFail']].merge(rh[keys+['PassOrFail']],on=keys,validate='one_to_one')
assert len(d)==574 and d.PassOrFail.sum()==8 and check.PassOrFail_x.equals(check.PassOrFail_y)
sensors=['Max_Injection_Pressure','Cushion_Position','Barrel_Temperature_5','Max_Injection_Speed']
segments={k:g.sort_values('TimeStamp').reset_index(drop=True) for k,g in observed.loc[observed['product'].eq('RG3')].groupby('history_segment')}
def corr(a,b):
    if len(a)<5 or not np.isfinite(a).all() or not np.isfinite(b).all() or np.std(a)<1e-8 or np.std(b)<1e-8: return np.nan
    return float(np.corrcoef(a,b)[0,1])
rows=[]
figdir=root/'reports/figures/rg3_shortshot_sequences'
figdir.mkdir(exist_ok=True)
for _,r in d.iterrows():
    g=segments[r.history_segment]
    i=int(np.searchsorted(g.TimeStamp.values,np.datetime64(r.TimeStamp)))
    z=g.iloc[max(0,i-10):i+1]
    assert z.TimeStamp.max()<=r.TimeStamp
    out={'TimeStamp':r.TimeStamp,'available_prior_shots':len(z)-1}
    dif=None
    if len(z)==11 and z[sensors].notna().all().all():
        dif=np.diff(z[sensors].to_numpy(float),axis=0)
        for j,c in enumerate(sensors[:3]):
            path=np.abs(dif[:,j]).sum()
            out[c+'_path_efficiency']=abs(dif[:,j].sum())/path if path>1e-8 else np.nan
        energy=dif[:,0]**2
        out['pressure_recent3_energy_share']=energy[-3:].sum()/energy.sum() if energy.sum()>1e-8 else np.nan
        out['pressure_cushion_delta_sync']=corr(dif[:,0],dif[:,1])
        # Fixed one-shot lag contrast; does not search for a best lag.
        out['pressure_then_cushion_lag_advantage']=corr(dif[:-1,0],dif[1:,1])-corr(dif[:-1,0],dif[:-1,1])
    # Next-shot recovery is explanatory only, never part of causal feature list.
    for c in sensors[:2]:
        j=sensors.index(c)
        out[c+'_jump']=r[c]-g.iloc[i-1][c] if i>=1 else np.nan
        out[c+'_next_recovery_fraction']=np.nan
        if i>=1 and i+1<len(g):
            a=g.iloc[i][c]-g.iloc[i-1][c]
            b=g.iloc[i+1][c]-g.iloc[i][c]
            if np.isfinite(a) and np.isfinite(b) and abs(a)>1e-8:
                out[c+'_next_recovery_fraction']=-b/a
    rows.append(out)
    if r.PassOrFail==1:
        q=g.iloc[max(0,i-10):min(len(g),i+4)]
        x=(q.TimeStamp-r.TimeStamp).dt.total_seconds()/60
        fig,axes=plt.subplots(3,1,figsize=(9,7),sharex=True)
        for ax,c in zip(axes,sensors[:3]):
            ax.plot(x,q[c],marker='o',ms=3,lw=1)
            ax.axvline(0,color='red',ls='--')
            ax.axvspan(0,max(float(x.max()),0),color='gray',alpha=.12)
            ax.set_ylabel(c,fontsize=8); ax.ticklabel_format(axis='y',style='plain',useOffset=False); ax.grid(alpha=.25)
        axes[-1].set_xlabel('Minutes relative to defective shot; shaded future is descriptive only')
        fig.suptitle('RG3 RH short shot '+str(r.TimeStamp))
        fig.tight_layout()
        fig.savefig(figdir/('case_'+r.TimeStamp.strftime('%Y%m%d_%H%M%S')+'.png'),dpi=140)
        plt.close(fig)
extra=pd.DataFrame(rows)
d=d.merge(extra,on='TimeStamp',validate='one_to_one')
causal=[c for c in extra.columns if c.endswith('_path_efficiency')]+['pressure_recent3_energy_share','pressure_cushion_delta_sync','pressure_then_cushion_lag_advantage']
comparisons=[]; cases=[]
for _,r in d.loc[d.PassOrFail.eq(1)].iterrows():
    normals=d.loc[d.comparison_block.eq(r.comparison_block)&d.PassOrFail.eq(0)]
    prior=normals.loc[normals.TimeStamp.lt(r.TimeStamp)]
    near=prior.iloc[-1] if len(prior) else None
    unusual=[]
    for c in causal:
        v=normals[c].dropna()
        lo,hi=v.quantile([.05,.95]) if len(v)>=10 else (np.nan,np.nan)
        rank=100*((v<r[c]).sum()+.5*(v==r[c]).sum())/len(v) if len(v) and pd.notna(r[c]) else np.nan
        outside=bool(pd.notna(r[c]) and pd.notna(lo) and (r[c]<lo or r[c]>hi))
        if outside: unusual.append(c)
        comparisons.append(dict(TimeStamp=r.TimeStamp,feature=c,defect_value=r[c],normal_count=len(v),normal_p05=lo,normal_p95=hi,normal_percentile=rank,outside_middle90=outside,nearest_prior_normal_value=near[c] if near is not None else np.nan))
    cases.append(dict(TimeStamp=r.TimeStamp,comparison_block=r.comparison_block,available_prior_shots=r.available_prior_shots,
                      nearest_prior_normal_timestamp=near.TimeStamp if near is not None else pd.NaT,
                      prior_normal_gap_minutes=(r.TimeStamp-near.TimeStamp).total_seconds()/60 if near is not None else np.nan,
                      unusual_features='; '.join(unusual) or ('none' if r.available_prior_shots>=10 else 'insufficient history'),
                      pressure_jump=r.Max_Injection_Pressure_jump,pressure_next_recovery_fraction=r.Max_Injection_Pressure_next_recovery_fraction,
                      cushion_jump=r.Cushion_Position_jump,cushion_next_recovery_fraction=r.Cushion_Position_next_recovery_fraction))
ct=pd.DataFrame(comparisons); cases=pd.DataFrame(cases)
d.to_csv(tables/'rg3_shortshot_sequence_features.csv',index=False,encoding='utf-8-sig')
ct.to_csv(tables/'rg3_shortshot_sequence_case_comparisons.csv',index=False,encoding='utf-8-sig')
cases.to_csv(tables/'rg3_shortshot_all8_sequence_review.csv',index=False,encoding='utf-8-sig')
summary=[]
for c in causal:
    validgas=int(d.loc[d.PassOrFail.eq(1),c].notna().sum())
    gasout=int(ct.loc[ct.feature.eq(c),'outside_middle90'].sum())
    nvalid=0; nout=0
    for _,g in d.loc[d.PassOrFail.eq(0)].groupby('comparison_block'):
        v=g[c].dropna()
        if len(v)<10: continue
        lo,hi=v.quantile([.05,.95]); nvalid+=len(v); nout+=int(((v<lo)|(v>hi)).sum())
    summary.append(dict(feature=c,defects_available=validgas,defects_outside=gasout,normals_available=nvalid,normals_outside=nout))
summary=pd.DataFrame(summary)
summary.to_csv(tables/'rg3_shortshot_sequence_pattern_summary.csv',index=False,encoding='utf-8-sig')
def md(frame):
    cols=frame.columns.tolist()
    result=['| '+' | '.join(cols)+' |','|'+'|'.join(['---']*len(cols))+'|']
    for _,r in frame.iterrows():
        result.append('| '+' | '.join(str(r[c]).replace('|','/') for c in cols)+' |')
    return result
union_flag=np.zeros(len(d),bool); union_known=np.zeros(len(d),bool)
for _,g in d.groupby('comparison_block'):
    normal=g.loc[g.PassOrFail.eq(0)]
    for c in causal:
        lo,hi=normal[c].quantile([.05,.95]); ids=g.index
        union_flag[ids]|=((g[c]<lo)|(g[c]>hi)).to_numpy(); union_known[ids]|=g[c].notna().to_numpy()
report=['# RG3 RH 미성형: 변화 순서·누적·엇박자 점검','',
'## 범위','',
'RG3 RH 정상566개 + 미성형8개. 가스17개 제외. 같은 원본 Shot 센서·RH 라벨 일치 검증. hayeon에서만 작업, 병합·커밋·푸시 없음.',
'','## 이전 실험과 무엇이 다른가','',
'- 기존 단순값, delta1, 과거 표준편차, 장기평균 대비 편차, 정상 회귀 잔차를 새 이름으로 반복하지 않았다.',
'- 단일센서 증가/감소의 경로 형태와 최근 변화 집중도, 두 센서 변화량의 동시/한 Shot 뒤 동조를 비교한다. 동일 센서를 쓰지만 계산하는 정보가 다르다.',
'- 새 피처6개는 탐색용 후보다. 이번에는 특징 확인이 먼저라는 합의대로 RF를 돌리거나 최적 조합을 고르지 않았다.',
'','## 정의','',
'- 경로 효율(압력·쿠션·배럴5 각각): 직전10개 변화의 순변화 절댓값 / 변화 절댓값 합. 1에 가까우면 한 방향으로 누적, 0에 가까우면 오르내림. 변화 크기 자체와는 다르며 일정하면 결측.',
'- 최근3개 압력 변화 집중도: 최근3개 변화량 제곱합 / 최근10개 변화량 제곱합. 최근에 움직임이 몰렸는지 확인.',
'- 압력·쿠션 동조: 최근10개 변화량의 상관계수. 양수는 함께 움직임, 음수는 반대로 움직임. 인과관계 아님.',
'- 압력 후 쿠션 지연 우세: 압력 변화와 다음 Shot 쿠션 변화의 상관 − 같은 Shot 변화 상관. 고정 한 Shot 간격만 사용하며 가장 유리한 지연을 탐색하지 않음.',
'- 예측 후보는 현재까지 센서만 사용. 미래3개는 그래프 설명용. 다음 Shot 회복률은 -다음변화/현재변화이며 1이면 원위치 회복, 현재 변화가 작으면 불안정하므로 독립 특징 근거로 채택하지 않음.',
'- 설비·제품·Timestamp 키 중복 정리 및 30분 이력 초기화 유지. 이 구간을 실제 Run으로 확정하지 않음.',
'','## 정상에서도 얼마나 흔한가','',
'각 불량의 같은 관측 구간 정상값 가운데90% 범위와 비교. 정상도 약10%가 범위 밖에 나오는 것이 자연스럽다. 정상 범위에 해당 구간 전체 정상(미래 포함)을 사용했으므로 설명용 비교이며 배포 가능한 판정 기준이나 모델 성능이 아니다. 6개 후보를 함께 찾으므로 우연한 이탈도 늘어난다.',
'']+md(summary)+['','## 미성형 8개 전체','']+md(cases)+[
'','## 이번 결과 해석','',
f'6개 중 하나라도 정상 범위 밖이라는 탐색 규칙을 적용하면 미성형 {int((union_flag & d.PassOrFail.eq(1)).sum())}/8개가 해당하지만 정상도 {int((union_flag & d.PassOrFail.eq(0)).sum())}/{int((union_known & d.PassOrFail.eq(0)).sum())}개가 해당한다. 검출 개선으로 주장할 수 없다.',
'압력·쿠션의 한 방향 누적은 미성형에서 정상 범위 밖인 사례가 각각0개였다. 배럴5 경로형태2개, 압력 최근변화 집중1개, 압력·쿠션 동조2개, 한 Shot 지연 대비3개가 정상 가운데90% 밖이었다. 가장 많이 걸린 지연 대비3개는 모두10/22에 있고 방향도 높은 쪽2개/낮은 쪽1개로 다르다. 10/23에서도 같은 패턴이 반복된다는 근거는 아직 없다.',
'따라서 지연 관계는 재검증 후보이지만 대표 미성형 신호로 확정하지 않는다. 다음Shot 회복은 설명용이며 예측에 사용하지 않는다. 이번에는 RF 학습 없이 특징을 점검했다.',
'','## 판단 기준','',
'한두 불량에서만 튀고 정상에서도 자주 보이는 패턴은 대표 특징이라고 하지 않는다. 충분한 이력이 없는 사례와 센서가 일정해 계산 불가능한 사례도 정상 패턴으로 처리하지 않는다. 여러 구간에서 같은 패턴이 반복되는지를 우선 확인하고, 이후 평가용 구간에서 학습 정상으로만 기준을 정해야 한다.',
'','## 산출물','',
'- notebooks/20_rg3_shortshot_sequence_patterns.ipynb',
'- reports/tables/rg3_shortshot_all8_sequence_review.csv: 모든8개 요약',
'- reports/tables/rg3_shortshot_sequence_case_comparisons.csv: 각 센서 관계 값·정상 범위·순위',
'- reports/figures/rg3_shortshot_sequences/: 불량8개별 과거·현재·설명용 미래 그래프',
]
(root/'reports/rg3_shortshot_sequence_review.md').write_text('\n'.join(report),encoding='utf-8')
print(summary.to_string(index=False))
print(cases.to_string(index=False))


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\hayeon\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


                               feature  defects_available  defects_outside  normals_available  normals_outside
Max_Injection_Pressure_path_efficiency                  8                0                557               24
      Cushion_Position_path_efficiency                  8                0                557               21
  Barrel_Temperature_5_path_efficiency                  8                2                557               33
         pressure_recent3_energy_share                  8                1                557               54
           pressure_cushion_delta_sync                  8                2                557               56
   pressure_then_cushion_lag_advantage                  8                3                557               58
          TimeStamp    comparison_block  available_prior_shots nearest_prior_normal_timestamp  prior_normal_gap_minutes                                                          unusual_features  pressure_jump  pressure_next